In [1]:
import os
import pickle
import joblib
import numpy as np
import pandas as pd
from collections.abc import Mapping, Sequence

# ====== 修改为你的 q_table 文件路径 ======
QTABLE_PATH = "./MacroHFT/result/high_level/ETHUSDT/long_action_base_202basetest5/q_table_ef6f4d2f87efe39daf41384f2f6cd72e.pkl"


def _safe_shape(x):
    try:
        return tuple(x.shape)
    except Exception:
        return None


def _num_stats(x: np.ndarray):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {"count": 0}
    return {
        "count": int(x.size),
        "mean": float(x.mean()),
        "std": float(x.std()),
        "min": float(x.min()),
        "p05": float(np.quantile(x, 0.05)),
        "p50": float(np.quantile(x, 0.50)),
        "p95": float(np.quantile(x, 0.95)),
        "max": float(x.max()),
    }


def summarize_obj(obj, name="root", max_depth=4, max_items=20, _depth=0):
    """递归打印/汇总结构：dict/list/np/torch/pandas，以及未知对象的 repr。
    返回：用于展示的扁平 rows（DataFrame）。
    """
    rows = []

    def add_row(path, typ, shape=None, extra=None):
        r = {"path": path, "type": typ, "shape": shape}
        if extra:
            r.update(extra)
        rows.append(r)

    if _depth > max_depth:
        add_row(name, type(obj).__name__, _safe_shape(obj), {"note": "max_depth_reached"})
        return rows

    # numpy
    if isinstance(obj, np.ndarray):
        st = _num_stats(obj) if obj.dtype.kind in "fc" else {"count": int(obj.size)}
        add_row(name, "np.ndarray", obj.shape, {"dtype": str(obj.dtype), **st})
        return rows

    # pandas
    if isinstance(obj, pd.DataFrame):
        add_row(name, "pd.DataFrame", obj.shape, {"columns": list(obj.columns)[:10]})
        return rows
    if isinstance(obj, pd.Series):
        add_row(name, "pd.Series", obj.shape, {"dtype": str(obj.dtype)})
        return rows

    # dict-like
    if isinstance(obj, Mapping):
        add_row(name, type(obj).__name__, None, {"n_keys": len(obj)})
        for i, (k, v) in enumerate(obj.items()):
            if i >= max_items:
                add_row(f"{name}", type(obj).__name__, None, {"note": f"truncated>{max_items}"})
                break
            rows.extend(summarize_obj(v, name=f"{name}[{repr(k)}]", max_depth=max_depth, max_items=max_items, _depth=_depth+1))
        return rows

    # list/tuple
    if isinstance(obj, (list, tuple)):
        add_row(name, type(obj).__name__, (len(obj),), None)
        for i, v in enumerate(obj[:max_items]):
            rows.extend(summarize_obj(v, name=f"{name}[{i}]", max_depth=max_depth, max_items=max_items, _depth=_depth+1))
        if len(obj) > max_items:
            add_row(name, type(obj).__name__, (len(obj),), {"note": f"truncated>{max_items}"})
        return rows

    # scalar number
    if isinstance(obj, (int, float, np.integer, np.floating)):
        add_row(name, type(obj).__name__, None, {"value": float(obj)})
        return rows

    # fallback
    add_row(name, type(obj).__name__, _safe_shape(obj), {"repr": repr(obj)[:200]})
    return rows


def load_qtable(path: str):
    # 先试 joblib（很多 repo 用 joblib.dump）
    try:
        return joblib.load(path)
    except Exception:
        pass
    # 再试 pickle
    with open(path, "rb") as f:
        return pickle.load(f)


assert os.path.exists(QTABLE_PATH), f"File not found: {QTABLE_PATH}"
qtable = load_qtable(QTABLE_PATH)

print("Loaded:", QTABLE_PATH)
print("Top-level type:", type(qtable))

rows = summarize_obj(qtable, max_depth=5, max_items=50)
df = pd.DataFrame(rows)

# 更易读：先按 path 长度排序
_df = df.copy()
_df["path_len"] = _df["path"].astype(str).map(len)
_df = _df.sort_values(["path_len", "path"]).drop(columns=["path_len"])

display(_df)

# 如果 top-level 是 dict，额外给一个 keys 总览
if isinstance(qtable, dict):
    print("\nTop-level keys:")
    print(list(qtable.keys()))


Loaded: ./MacroHFT/result/high_level/ETHUSDT/long_action_base_202basetest5/q_table_ef6f4d2f87efe39daf41384f2f6cd72e.pkl
Top-level type: <class 'numpy.ndarray'>


,path,type,shape,dtype,count,mean,std,min,p05,p50,p95,max
0,root,np.ndarray,"(525240, 2, 2)",float64,2100960,4.98469,5.313183,-4.653519,0.221017,3.692605,13.958382,93.500344


In [3]:
# ====== 2) 按 q_table 的真实三维语义做统计（避免全局 flatten 失真） ======
# qtable shape: [T, C, A] = [525240, 2, 2]
# 这里给出：
# - 每个 (clf, action) 的分布统计
# - 每个时间 t 的 maxQ/minQ 分布（看是否有极端尖峰）
# - 极值点定位：max 出现在 (t, clf, action) 哪
# - 时间分段统计：前/中/后是否漂移

assert isinstance(qtable, np.ndarray) and qtable.ndim == 3, f"Expect ndarray [T,C,A], got {type(qtable)} shape={getattr(qtable,'shape',None)}"
T, C, A = qtable.shape
print(f"qtable shape: T={T}, C={C}, A={A}")

clf_names = ["slope", "vol"] if C == 2 else [f"clf_{i}" for i in range(C)]
a_names = [f"a{i}" for i in range(A)]

# 1) 每个 (clf, action) 分布统计
rows_ca = []
for ci in range(C):
    for ai in range(A):
        x = qtable[:, ci, ai]
        x = x[np.isfinite(x)]
        rows_ca.append({
            "clf": clf_names[ci],
            "action": a_names[ai],
            **_num_stats(x),
        })

df_ca = pd.DataFrame(rows_ca).sort_values(["clf", "action"])
print("\n(A) per-(clf,action) stats:")
display(df_ca)

# 2) 每个时间 t 的 max/min（跨 clf/action）—— 检查是否少量极端点拉高 max
qt_max = np.nanmax(qtable, axis=(1, 2))  # [T]
qt_min = np.nanmin(qtable, axis=(1, 2))  # [T]
print("\n(B) per-t maxQ/minQ summary:")
display(pd.DataFrame({
    "metric": ["max_over_(c,a)", "min_over_(c,a)"],
    "mean": [qt_max.mean(), qt_min.mean()],
    "std": [qt_max.std(), qt_min.std()],
    "p50": [np.quantile(qt_max, 0.5), np.quantile(qt_min, 0.5)],
    "p95": [np.quantile(qt_max, 0.95), np.quantile(qt_min, 0.95)],
    "p99": [np.quantile(qt_max, 0.99), np.quantile(qt_min, 0.99)],
    "max": [qt_max.max(), qt_min.max()],
    "min": [qt_max.min(), qt_min.min()],
}))

# 3) 全局极值点定位（告诉你哪个 t/clf/action 最极端）
flat_argmax = int(np.nanargmax(qtable))
flat_argmin = int(np.nanargmin(qtable))
idx_max = np.unravel_index(flat_argmax, qtable.shape)
idx_min = np.unravel_index(flat_argmin, qtable.shape)
print("\n(C) global extreme locations:")
print("max:", float(np.nanmax(qtable)), "at (t,clf,action)=", idx_max, "->", clf_names[idx_max[1]], a_names[idx_max[2]])
print("min:", float(np.nanmin(qtable)), "at (t,clf,action)=", idx_min, "->", clf_names[idx_min[1]], a_names[idx_min[2]])

# 打印这些极值 t 周围的局部切片，看看是不是尖峰/脏点
win = 3
for tag, (t0, c0, a0) in [("max", idx_max), ("min", idx_min)]:
    lo = max(0, t0 - win)
    hi = min(T, t0 + win + 1)
    sl = qtable[lo:hi]  # [W,C,A]
    print(f"\n[{tag}] neighborhood t in [{lo},{hi}):")
    # 展示成 DataFrame：每行一个 t，列是 (clf,action)
    cols = []
    mat = []
    for ci in range(C):
        for ai in range(A):
            cols.append(f"{clf_names[ci]}_{a_names[ai]}")
    for tt in range(lo, hi):
        mat.append([qtable[tt, ci, ai] for ci in range(C) for ai in range(A)])
    display(pd.DataFrame(mat, columns=cols).assign(t=np.arange(lo, hi)).set_index("t"))

# 4) 时间分段统计：前/中/后（看漂移）
segments = {
    "head": slice(0, T//10),
    "mid": slice(T//2 - T//20, T//2 + T//20),
    "tail": slice(T - T//10, T),
}
rows_seg = []
for seg_name, sl in segments.items():
    for ci in range(C):
        for ai in range(A):
            x = qtable[sl, ci, ai]
            x = x[np.isfinite(x)]
            st = _num_stats(x)
            rows_seg.append({
                "segment": seg_name,
                "clf": clf_names[ci],
                "action": a_names[ai],
                "mean": st.get("mean"),
                "std": st.get("std"),
                "p05": st.get("p05"),
                "p50": st.get("p50"),
                "p95": st.get("p95"),
                "min": st.get("min"),
                "max": st.get("max"),
            })

df_seg = pd.DataFrame(rows_seg)
print("\n(D) segment drift check (head/mid/tail):")
display(df_seg.sort_values(["clf", "action", "segment"]))


qtable shape: T=525240, C=2, A=2

(A) per-(clf,action) stats:


,clf,action,count,mean,std,min,p05,p50,p95,max
0,slope,a0,525240,5.171680,5.315344,0.000000,0.383400,3.889479,14.141561,92.143555
1,slope,a1,525240,4.797699,5.304599,-4.653519,0.073309,3.493057,13.752917,93.074298
2,vol,a0,525240,4.800983,5.280358,-0.440132,0.115489,3.506049,13.688318,91.708087
3,vol,a1,525240,5.168397,5.339326,-4.062791,0.346589,3.875715,14.213447,93.500344



(B) per-t maxQ/minQ summary:


,metric,mean,std,p50,p95,p99,max,min
0,"max_over_(c,a)",5.345858,5.396775,4.051388,14.473415,26.687130,93.500344,0.000000
1,"min_over_(c,a)",4.623521,5.217411,3.332109,13.414160,25.505275,86.493680,-4.653519



(C) global extreme locations:
max: 93.50034396112761 at (t,clf,action)= (142971, 1, 1) -> vol a1
min: -4.653519134973093 at (t,clf,action)= (109338, 0, 1) -> slope a1

[max] neighborhood t in [142968,142975):


,slope_a0,slope_a1,vol_a0,vol_a1
t,,,,
142968,90.309898,86.380587,89.867520,86.822965
142969,91.222119,86.493680,90.782793,86.933006
142970,92.143555,82.707873,91.708087,83.143341
142971,85.142710,93.074298,84.716664,93.500344
142972,77.377748,86.002737,76.943774,86.436711
142973,78.159342,71.446610,77.715870,71.890082
142974,78.948830,77.340665,78.511194,77.778301



[min] neighborhood t in [109335,109342):


,slope_a0,slope_a1,vol_a0,vol_a1
t,,,,
109335,4.401747,2.164788,3.808779,2.757756
109336,4.446209,3.066508,3.854299,3.658418
109337,4.491120,2.334591,3.899412,2.926299
109338,4.536485,-4.653519,3.945757,-4.062791
109339,4.582308,4.257731,4.000434,4.839605
109340,4.628594,3.489646,4.045912,4.072328
109341,3.865160,4.675347,3.283610,5.256897



(D) segment drift check (head/mid/tail):


,segment,clf,action,mean,std,p05,p50,p95,min,max
0,head,slope,a0,9.478160,5.442413,3.274454,8.282415,20.019917,1.299880,47.362785
4,mid,slope,a0,6.121714,3.652400,2.025030,5.396820,12.509631,0.793333,34.295864
8,tail,slope,a0,1.696290,2.031774,0.066062,1.063659,5.078118,0.000000,27.392568
1,head,slope,a1,8.909402,5.523927,2.564901,7.695216,19.568625,-0.305014,47.841197
5,mid,slope,a1,5.789587,3.710681,1.629358,5.063351,12.235229,-0.533639,34.642287
9,tail,slope,a1,1.417109,2.053985,-0.257869,0.777078,4.838446,-1.707359,27.669261
2,head,vol,a0,8.912103,5.452647,2.701166,7.713253,19.490637,0.766552,46.874601
6,mid,vol,a0,5.788989,3.662889,1.681568,5.063484,12.199541,0.451381,33.996136
10,tail,vol,a0,1.412112,2.017392,-0.178979,0.770703,4.773667,-0.317508,27.097786
3,head,vol,a1,9.475459,5.513764,3.142320,8.262371,20.092059,0.223596,48.328493
